# SBA Charge-Off Review Knowledge Graph

This notebook uses the KMDS 0.5 notebook interface to record the reasoning, evidence, assumptions, transformations, experiments, and decisions in the SBA modeling workflow. The graph is intentionally broad: it contains every observation type supported by KMDS 0.5 rather than only final model results.

The target is a practical human-review queue for potentially distressed SBA loans. The graph captures why feature selection matters, how the canonical data partitions are used, how the model and operating cutoff were selected, and where the final test evidence sits.

In [1]:
from importlib.metadata import version
from pathlib import Path

import pandas as pd
from kmds import KnowledgeGraph

workspace_root = Path.cwd()
if workspace_root.name == "notebooks":
    workspace_root = workspace_root.parent

assert version("kmds") == "0.5.0", "Run this notebook with kmds 0.5.0"
graph = KnowledgeGraph("sba_chargeoff_review")
print(f"KMDS version: {version('kmds')}")
print(f"Workspace: {workspace_root}")

KMDS version: 0.5.0
Workspace: /home/rajiv/programming/kmds_migration/sba_migration


## Discover the observation vocabulary

The registry below drives both validation and insertion. This notebook uses all listed types so the result illustrates exploratory, representation, modeling-choice, model-selection, and experimental observations.

In [2]:
observation_types = graph.summarize_observation_types()
observation_types

                 observation_type              family                                                                                purpose
            Relevance Observation         exploratory           Record which data, variables, and analytical scope are relevant to the task.
         Data Quality Observation         exploratory       Document data defects, uncertainty, bias, drift, or representativeness concerns.
  Feature Engineering Observation data_representation    Capture the rationale for derived features, encodings, and learned representations.
  Data Transformation Observation data_representation       Record cleaning, scaling, reshaping, imputation, and other data transformations.
     Modelling Choice Observation    modelling_choice Explain why a model family, algorithm, configuration, or learning approach was chosen.
 Modelling Assumption Observation    modelling_choice   Preserve assumptions about the data-generating process, model, and learning setting.
        Model

,observation_type,family,purpose
0,Relevance Observation,exploratory,"Record which data, variables, and analytical s..."
1,Data Quality Observation,exploratory,"Document data defects, uncertainty, bias, drif..."
2,Feature Engineering Observation,data_representation,"Capture the rationale for derived features, en..."
3,Data Transformation Observation,data_representation,"Record cleaning, scaling, reshaping, imputatio..."
4,Modelling Choice Observation,modelling_choice,"Explain why a model family, algorithm, configu..."
5,Modelling Assumption Observation,modelling_choice,Preserve assumptions about the data-generating...
6,Model Selection Statement,model_selection,State the intended model-selection decision or...
7,Model Selection Setup Description,model_selection,"Describe evaluation data, metrics, baselines, ..."
8,Model Selection Observation,model_selection,Capture evidence and findings produced while c...
9,Model Selection Result Summary,model_selection,Summarize the selected model and the measured ...


## Project evidence and implementation map

The observations below are grounded in the following project artifacts:

- [Data cleaning and terminal-outcome preparation](clean_sba_dataset.ipynb)
- [Geographic feature derivation](clustering_feature_derivation.ipynb)
- [Feature-advisor example](feature_advisor_sba_example.ipynb)
- [Canonical featurization](sba_featurization.ipynb)
- [Training, calibration, cutoff selection, and test evaluation](sba_modeling.ipynb)
- [Modeling requirements](../documents/sba_modeling_requirements.md)
- [KMDS workflow overview](../documents/sba_modelling_overview.md)
- [Performance summary](../agent_documents/sba_performance_report_summary.md)

The SBA source is [Data.gov: SBA](https://data.gov/metrics/organization/sba/). This project evaluates only observed terminal outcomes and predicts the probability of charge-off for human review; it does not predict the charged-off dollar amount.

In [3]:
observations = [
    (
        "The workflow uses the SBA loan dataset published through Data.gov to estimate charge-off risk and refer high-risk loans to human review. Implementation: notebooks/clean_sba_dataset.ipynb.",
        "Relevance Observation",
    ),
    (
        "The evaluation population retains observed PIF and CHGOFF outcomes and excludes active EXEMPT, undisbursed COMMIT, and cancelled CANCLD records because their terminal outcomes are unavailable or not comparable.",
        "Data Quality Observation",
    ),
    (
        "Charged-off loans are a minority class. Accuracy alone can therefore hide a weak screening queue, so ranking metrics and review-oriented gains, lift, precision, and recall are required.",
        "Data Quality Observation",
    ),
    (
        "The final feature contract contains bankname, bankcity, initialinterestrate, bankstreet, and terminmonths. The feature set is a controlled five-feature judgment informed by the imbalanced feature screener; implementation: notebooks/sba_featurization.ipynb.",
        "Feature Engineering Observation",
    ),
    (
        "Geographic and clustered borrower features were investigated in notebooks/clustering_feature_derivation.ipynb, but the frozen modeling contract is limited to the five screened fields. Feature additions require a new controlled experiment.",
        "Feature Engineering Observation",
    ),
    (
        "Canonical 80/10/10 train, validation, and test assignments are persisted in dataset_partition and are never recreated inside modeling. Numeric median imputation and categorical M-estimate encoding are fit on train only and reused for validation and test.",
        "Data Transformation Observation",
    ),
    (
        "Gradient Boosting and Random Forest are the candidate model families. Isotonic calibration is required to improve the probability ranking used for screening and cumulative-gains analysis.",
        "Modelling Choice Observation",
    ),
    (
        "The modeling formulation assumes the selected fields are available at decision time, a frozen train/validation/test contract represents the deployment ordering of information, and human review remains the final decision for referred loans.",
        "Modelling Assumption Observation",
    ),
    (
        "Select the first-stage champion by validation PR-AUC, use validation only for the recall-first diagnostic and elbow cutoff, then apply the frozen model and cutoff once to test. Implementation: notebooks/sba_modeling.ipynb.",
        "Model Selection Statement",
    ),
    (
        "The comparison setup uses the canonical partitions from data/featurization/model_ready_numeric_data.csv. Candidates are Gradient Boosting and Random Forest, probabilities receive isotonic calibration, validation PR-AUC drives champion selection, and test is not used for tuning.",
        "Model Selection Setup Description",
    ),
    (
        "On validation, Gradient Boosting achieved PR-AUC 0.7560 and Random Forest achieved PR-AUC 0.7237. The broader feature set achieved validation PR-AUC 0.1673, evidence that candidate feature choice materially affects ranking quality.",
        "Model Selection Observation",
    ),
    (
        "Gradient Boosting was selected as the champion. The validation maximum-distance elbow selected the top 20.00% cohort and the frozen calibrated probability threshold 0.069643; score ties expanded the realized review population to 20.55% on validation.",
        "Model Selection Result Summary",
    ),
    (
        "Hypothesis: selecting a compact, multi-perspective feature set will produce materially better probability ranking and a usable human-review queue than fitting on the broad, noisy feature universe.",
        "Hypothesis Statement",
    ),
    (
        "The five-feature experiment increased test PR-AUC from 0.1523 to 0.7421, reduced the test review rate from 100.0% to 19.55%, and increased test lift from 1.00 to 4.22 relative to the prior broader feature set.",
        "Experimental Observation",
    ),
    (
        "The evidence supports the conjecture that redundant, sparse, correlated, or weakly measured fields obscure stable signal in an imbalanced dataset, while complementary feature-evaluation perspectives reduce dependence on one fitted classifier's importance scores.",
        "Experimental Conjecture",
    ),
    (
        "On held-out test, the frozen policy reviewed 19.55% of 7,510 loans, achieved precision 38.76%, recall 82.46%, F1 52.73%, PR-AUC 0.7421, balanced accuracy 0.8464, and lift 4.22. Results support a review queue, not automatic adjudication; future top-k and feature-membership experiments must preserve the canonical splits and rerun validation tuning.",
        "Experimental Result Summary",
    ),
    (
        "Two alternative formulations were considered before the final review model: cascading classifiers that optimize recall before precision, and utility scoring that compares predicted default probability and amount at risk against review versus no-review actions. They were not selected for the final baseline.",
        "Experimental Observation",
    ),
]

for text, observation_type in observations:
    graph.add_observation(text, observation_type)

print(f"Added {len(observations)} observations across {len(set(kind for _, kind in observations))} types.")

Added 17 observations across 14 types.


In [4]:
knowledge_table = graph.table()
knowledge_table

,sequence,observation_type,family,intent,text
0,1,Relevance Observation,exploratory,DATA_UNDERSTANDING,The workflow uses the SBA loan dataset publish...
1,2,Data Quality Observation,exploratory,DATA_UNDERSTANDING,The evaluation population retains observed PIF...
2,3,Data Quality Observation,exploratory,DATA_UNDERSTANDING,Charged-off loans are a minority class. Accura...
3,4,Feature Engineering Observation,data_representation,FEATURE_ASSESSMENT,"The final feature contract contains bankname, ..."
4,5,Feature Engineering Observation,data_representation,FEATURE_ASSESSMENT,Geographic and clustered borrower features wer...
5,6,Data Transformation Observation,data_representation,FEATURE_ASSESSMENT,"Canonical 80/10/10 train, validation, and test..."
6,7,Modelling Choice Observation,modelling_choice,MODEL_EXPLANATION,Gradient Boosting and Random Forest are the ca...
7,8,Modelling Assumption Observation,modelling_choice,MODEL_EXPLANATION,The modeling formulation assumes the selected ...
8,9,Model Selection Statement,model_selection,MODEL_SELECTION,Select the first-stage champion by validation ...
9,10,Model Selection Setup Description,model_selection,MODEL_SELECTION,The comparison setup uses the canonical partit...


## Feature-selection foundations

The graph's emphasis on controlled, evidence-based feature selection is consistent with established machine-learning guidance:

1. Domingos (2012), [A Few Useful Things to Know about Machine Learning](https://doi.org/10.1145/2347736.2347755), emphasizes that representation, feature engineering, and avoiding overfitting can matter as much as algorithm choice.
2. Guyon and Elisseeff (2003), [An Introduction to Variable and Feature Selection](https://www.jmlr.org/papers/v3/guyon03a.html), explains why feature selection should combine model quality with complementary evaluation methods rather than rely on one importance ranking.
3. Kuhn and Johnson (2013), [Applied Predictive Modeling](https://doi.org/10.1007/978-1-4614-6849-3), provides practical guidance for preprocessing, feature selection, resampling, and predictive-model evaluation.
4. He and Garcia (2009), [Learning from Imbalanced Data](https://doi.org/10.1109/TKDE.2008.239), covers the evaluation and learning challenges created by skewed class distributions.
5. The project's [imbalanced_feature_screener](https://github.com/rajivsam/imbalanced_feature_screener) applies multiple feature-evaluation perspectives to this imbalanced SBA problem.

In [5]:
supported_types = set(observation_types["observation_type"])
used_types = set(knowledge_table["observation_type"])
missing_types = supported_types - used_types
unexpected_types = used_types - supported_types

assert not missing_types, f"Missing observation types: {sorted(missing_types)}"
assert not unexpected_types, f"Unexpected observation types: {sorted(unexpected_types)}"
assert len(used_types) == 14

output_path = workspace_root / "data" / "kmds" / "sba_chargeoff_review.xml"
output_path.parent.mkdir(parents=True, exist_ok=True)
saved_path = graph.save(output_path)

restored = KnowledgeGraph.load(saved_path)
restored_table = restored.table()
pd.testing.assert_frame_equal(knowledge_table, restored_table)

print(f"Saved: {saved_path}")
print(f"Verified {len(used_types)} observation types and {len(restored_table)} observations after reload.")
restored_table

Saved: /home/rajiv/programming/kmds_migration/sba_migration/data/kmds/sba_chargeoff_review.xml
Verified 14 observation types and 17 observations after reload.


,sequence,observation_type,family,intent,text
0,1,Relevance Observation,exploratory,DATA_UNDERSTANDING,The workflow uses the SBA loan dataset publish...
1,2,Data Quality Observation,exploratory,DATA_UNDERSTANDING,The evaluation population retains observed PIF...
2,3,Data Quality Observation,exploratory,DATA_UNDERSTANDING,Charged-off loans are a minority class. Accura...
3,4,Feature Engineering Observation,data_representation,FEATURE_ASSESSMENT,"The final feature contract contains bankname, ..."
4,5,Feature Engineering Observation,data_representation,FEATURE_ASSESSMENT,Geographic and clustered borrower features wer...
5,6,Data Transformation Observation,data_representation,FEATURE_ASSESSMENT,"Canonical 80/10/10 train, validation, and test..."
6,7,Modelling Choice Observation,modelling_choice,MODEL_EXPLANATION,Gradient Boosting and Random Forest are the ca...
7,8,Modelling Assumption Observation,modelling_choice,MODEL_EXPLANATION,The modeling formulation assumes the selected ...
8,9,Model Selection Statement,model_selection,MODEL_SELECTION,Select the first-stage champion by validation ...
9,10,Model Selection Setup Description,model_selection,MODEL_SELECTION,The comparison setup uses the canonical partit...
